# 08 · Failure Analysis of the External Performance Drop

Why does balanced accuracy fall from about 75% on HAM10000 to below 50% on ISIC images? Each experiment below uses external labels
**only for measurement**; nothing is tuned on them.

1. **Verification:** label mapping and a search for HAM10000 photographs duplicated in the external sample.
2. **Description:** prevalence of black dermoscope borders and accuracy with and without them.
3. **Border crop:** crop detected borders to the largest centred square inside the field of view (rule fixed in advance) and
   re-run the unchanged models.
4. **Colour constancy:** Shades-of-Gray normalisation of every image.
5. **Prior shift:** expectation–maximisation adjustment of the class priors (label-free).
6. **Features versus decision layer:** linear probes on frozen ensemble features, trained on HAM10000 or, purely as a diagnostic,
   on external labels with nested lesion-grouped cross-validation.

Differences are reported as the mean over the five fold-ensembles with 95% confidence intervals from a lesion-cluster bootstrap.

**Inputs:** notebook 07 caches, checkpoints · **Outputs:** `results/border_crop_results.csv`, tables in `outputs/failure_analysis/`

## Setup

Paths can be edited in the next cell or set as environment variables (`HAM_PROJECT_OUT`, `HAM10000_DIR`, `ISIC2019_DIR`).

In [ ]:
import os, json, math, time, random, platform, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                                   # progress bars are optional
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

In [ ]:
# ---------------------------------------------------------------- paths (edit here or set the environment variables)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

def env_path(name, default):
    return Path(os.environ.get(name, str(default))).expanduser()

PROJECT_OUT  = env_path('HAM_PROJECT_OUT', r'D:/HAM_PROJECT/outputs_v2')   # caches, checkpoints, features, tables (not in Git)
HAM_DIR      = env_path('HAM10000_DIR', r'D:/HAM_PROJECT/dataset')      # HAM10000 images, metadata and lesion masks
ISIC2019_DIR = env_path('ISIC2019_DIR', r'D:/Skin Dataset/ISIC2019')    # ISIC 2019 training images, ground truth, metadata
RESULTS_DIR  = env_path('RESULTS_DIR', REPO / 'results')             # tables and figures shown in the README
SPLITS_DIR   = env_path('SPLITS_DIR', REPO / 'splits')               # published fold assignment

# ---------------------------------------------------------------- study settings (keep unchanged to reproduce the results)
CLASSES   = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
MALIGNANT = ['akiec', 'bcc', 'mel']                                  # (pre)malignant classes
MODELS    = ['efficientnet_b4', 'convnext_base', 'densenet121']
LABEL_OF  = {'efficientnet_b4': 'EfficientNet-B4', 'convnext_base': 'ConvNeXt-Base', 'densenet121': 'DenseNet-121'}
IMG_SIZE, SEED, N_FOLDS = 224, 42, 5
DROPOUT, MC_SAMPLES = 0.4, 30                                       # head dropout; MC Dropout passes per model
UNC_QUANTILE, GMM_N, N_NVREF = 0.75, 3, 1000                        # uncertainty flag; HBDS levels; nevus reference size
VARIANT = 'focal_ls'                                                # main models (focal loss + label smoothing)
EXT_MC_SEED0 = 555                                                  # MC seed offset for external data

C = len(CLASSES)
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
MAL_IDX = [CLASS2IDX[c] for c in MALIGNANT]
EPS = 1e-12
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DIRS = {k: PROJECT_OUT / k for k in ['splits', 'checkpoints', 'cache', 'tables', 'figures']}
for _d in list(DIRS.values()) + [RESULTS_DIR, SPLITS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def mean_sd(values, digits=4):
    s = pd.Series(values, dtype=float).dropna()
    return f'{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}' if len(s) > 1 else (f'{s.mean():.{digits}f}' if len(s) else 'n/a')

print(f'Python {platform.python_version()} | PyTorch {torch.__version__} | device: {DEVICE}')
print(f'Project outputs: {PROJECT_OUT}')

In [ ]:
import cv2
from PIL import Image

def find_file(root, name):
    """Locate a dataset file by name anywhere below root."""
    direct = Path(root) / name
    if direct.is_file():
        return direct
    hits = sorted(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under {root}. See the Dataset section of README.md.")
    return hits[0]

def image_index(root, exclude=('segmentation', 'mask')):
    """Map image stem -> path for all images below root (lesion masks excluded)."""
    index = {}
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and not any(e in p.as_posix().lower() for e in exclude):
            index.setdefault(p.stem, p)
    return index

def read_rgb(path):
    with Image.open(path) as im:
        return np.array(im.convert('RGB'))

def read_rgb_224(path):
    """Pipeline image loading: PIL -> RGB -> cv2 bilinear resize to 224 x 224."""
    return cv2.resize(read_rgb(path), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

def build_image_cache(paths, npy_path, json_path, desc='caching images'):
    """Decode every image once into a uint8 array (N, 224, 224, 3). Reused if it already exists."""
    paths = [str(p) for p in paths]
    if Path(npy_path).exists() and Path(json_path).exists():
        cached = json.load(open(json_path))
        if [Path(p).stem for p in cached] == [Path(p).stem for p in paths]:
            return np.load(npy_path, mmap_mode='r')
        raise RuntimeError(f'{json_path} exists but lists different images - delete it to rebuild the cache.')
    Path(npy_path).parent.mkdir(parents=True, exist_ok=True)
    arr = np.lib.format.open_memmap(npy_path, mode='w+', dtype=np.uint8, shape=(len(paths), IMG_SIZE, IMG_SIZE, 3))
    for i, p in enumerate(tqdm(paths, desc=desc)):
        arr[i] = read_rgb_224(p)
    arr.flush(); del arr
    json.dump(paths, open(json_path, 'w'))
    return np.load(npy_path, mmap_mode='r')

def load_ham():
    """HAM10000 metadata in the order of the image cache, labels and lesion groups."""
    meta = pd.read_csv(find_file(HAM_DIR, 'HAM10000_metadata.csv')).drop_duplicates('image_id').set_index('image_id')
    order_file = DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.json'
    if not order_file.exists():
        raise FileNotFoundError(f'{order_file} not found - run notebook 01 first.')
    ids = [Path(p).stem for p in json.load(open(order_file))]
    ham = meta.loc[ids].reset_index()
    return ham, ham['dx'].map(CLASS2IDX).astype(int).values, ham['lesion_id'].astype(str).values

def load_splits():
    f = DIRS['splits'] / 'cv_splits.json'
    if not f.exists():
        raise FileNotFoundError(f'{f} not found - run notebook 01 first.')
    return {int(k): {kk: np.asarray(vv, dtype=int) for kk, vv in v.items()} for k, v in json.load(open(f)).items()}

In [ ]:
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

def make_head(in_dim, p=DROPOUT):
    """LayerNorm -> Dropout(0.4) -> Linear(512) -> GELU -> Dropout(0.2) -> Linear(7)."""
    return nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(p), nn.Linear(in_dim, 512),
                         nn.GELU(), nn.Dropout(p / 2), nn.Linear(512, C))

class SkinLesionModel(nn.Module):
    """ImageNet-pretrained timm backbone with global average pooling + classification head."""
    def __init__(self, model_name, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.feat_dim = self.backbone.num_features
        self.classifier = make_head(self.feat_dim)

    def forward_features(self, x):
        return self.backbone(x)

    def forward(self, x):
        return self.classifier(self.backbone(x))

def ckpt_path(k, m, variant=VARIANT):
    return DIRS['checkpoints'] / f'fold{k}' / variant / f'{m}.pth'

def load_model(k, m, variant=VARIANT):
    p = ckpt_path(k, m, variant)
    if not p.exists():
        raise FileNotFoundError(f'Checkpoint {p} not found - train it with notebook 02 or download the released weights.')
    ck = torch.load(p, map_location='cpu', weights_only=True)
    model = SkinLesionModel(m)
    model.load_state_dict(ck['state_dict'] if 'state_dict' in ck else ck, strict=True)
    model.to(DEVICE).eval()
    for q in model.parameters():
        q.requires_grad_(False)
    return model

INFER_BATCH = 64 if DEVICE.type == 'cuda' else 16

@torch.no_grad()
def extract_features(model, images, rows=None, batch_size=INFER_BATCH):
    """Pooled backbone features (eval mode, float16 autocast on GPU), returned as float16 as in the study."""
    rows = np.arange(len(images)) if rows is None else np.asarray(rows)
    out = []
    for s in range(0, len(rows), batch_size):
        x = torch.stack([eval_transform(image=np.ascontiguousarray(images[i]))['image'] for i in rows[s:s + batch_size]]).to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            f = model.forward_features(x)
        out.append(f.float().cpu().numpy().astype(np.float16))
    return np.concatenate(out) if out else np.zeros((0, model.feat_dim), np.float16)

In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.covariance import LedoitWolf
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

def fold_dir(k, variant=VARIANT):
    return DIRS['cache'] / f'fold{k}' / variant

def feat_path(k, m, variant=VARIANT):
    return fold_dir(k, variant) / f'{m}_features.npz'

def head_path(k, m, variant=VARIANT):
    return fold_dir(k, variant) / f'{m}_head.pt'

def ensure_fold_cache(k, images, labels, splits, variant=VARIANT):
    """Save each model's head and its validation, test and nevus-reference features (skipped if already cached)."""
    for m in MODELS:
        if feat_path(k, m, variant).exists() and head_path(k, m, variant).exists():
            continue
        fold_dir(k, variant).mkdir(parents=True, exist_ok=True)
        model = load_model(k, m, variant)
        torch.save({'in_dim': model.feat_dim, 'state': model.classifier.state_dict()}, head_path(k, m, variant))
        sp = splits[k]
        nv_train = sp['train'][labels[sp['train']] == CLASS2IDX['nv']]
        nvref_idx = np.sort(np.random.default_rng(SEED + k).choice(nv_train, size=min(N_NVREF, len(nv_train)), replace=False))
        np.savez(feat_path(k, m, variant), val=extract_features(model, images, sp['val']),
                 test=extract_features(model, images, sp['test']), nvref=extract_features(model, images, nvref_idx),
                 nvref_idx=nvref_idx)
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

def load_head(k, m, variant=VARIANT):
    d = torch.load(head_path(k, m, variant), map_location='cpu', weights_only=True)
    h = make_head(d['in_dim']); h.load_state_dict(d['state'])
    return h.to(DEVICE).eval()

def softmax_np(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

@torch.no_grad()
def head_logits(head, feats, chunk=8192):
    head.eval(); out = []
    for i in range(0, len(feats), chunk):
        out.append(head(torch.from_numpy(np.asarray(feats[i:i + chunk], np.float32)).to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def head_mc(head, feats, n, seed, chunk=8192):
    """MC Dropout on the head only (the backbones contain no dropout, so this equals full stochastic passes)."""
    head.train(); torch.manual_seed(seed)
    X = torch.from_numpy(np.asarray(feats, np.float32)).to(DEVICE)
    out = np.empty((n, len(feats), C), np.float32)
    for s in range(n):
        for i in range(0, len(feats), chunk):
            out[s, i:i + chunk] = F.softmax(head(X[i:i + chunk]), 1).cpu().numpy()
    head.eval()
    return out

def mc_stats(samples):
    pbar = samples.mean(0); chat = pbar.argmax(1)
    var = samples[:, np.arange(samples.shape[1]), chat].var(0)            # variance of the predicted-class probability
    ent = -(pbar * np.log(pbar + EPS)).sum(1)
    exp_ent = -(samples * np.log(samples + EPS)).sum(2).mean(0)
    return {'probs': pbar, 'var': var, 'entropy': ent, 'mi': ent - exp_ent}

def nll(p, y):
    return float(-np.mean(np.log(p[np.arange(len(y)), y] + EPS)))

def fit_temperature(logits, y):
    lg = np.asarray(logits, np.float64)
    return float(minimize_scalar(lambda T: nll(softmax_np(lg / T), y), bounds=(0.05, 10.0), method='bounded').x)

def mahalanobis(X, mu, prec, chunk=4096):
    out = np.empty(len(X))
    for i in range(0, len(X), chunk):
        d = np.asarray(X[i:i + chunk], np.float64) - mu
        out[i:i + chunk] = np.sqrt(np.maximum(np.sum((d @ prec) * d, 1), 0))
    return out

def fit_gmm(scores, k):
    g = GaussianMixture(n_components=k, covariance_type='full', n_init=10, max_iter=300, random_state=SEED).fit(scores.reshape(-1, 1))
    order = np.argsort(g.means_.ravel()); rank = np.empty(k, int); rank[order] = np.arange(k)
    return g, rank

def gmm_levels(g, rank, s):
    return rank[g.predict(np.asarray(s).reshape(-1, 1))]

def _value(v, k):
    return v(k) if callable(v) else v

def build_fold(k, ext=None, variant=VARIANT):
    """Rebuild the full framework of fold k from cached features.

    ext: optional {name: dict(feats=fn(k, m), y=..., ymal=..., groups=...)} for additional (external) image sets.
    Every fitted quantity (temperatures, HBDS reference and scaling, GMM levels, uncertainty threshold) uses the
    HAM10000 validation partition of fold k only.
    """
    ext = ext or {}
    sp = SPLITS[k]; base = SEED * 1000 + k; y_val = LABELS[sp['val']]
    parts = ['val', 'test'] + list(ext)
    feats = {m: {} for m in MODELS}
    for m in MODELS:
        z = np.load(feat_path(k, m, variant))
        feats[m].update(val=z['val'], test=z['test'], nvref=z['nvref'])
        for ds, d in ext.items():
            feats[m][ds] = d['feats'](k, m)
    R = {p: {'single_det': {}, 'single_cal': {}} for p in parts}
    samples = {p: [] for p in parts}; Ts = {}
    for mi, m in enumerate(MODELS):
        h = load_head(k, m, variant)
        lg = {p: head_logits(h, feats[m][p]) for p in parts}
        Ts[m] = fit_temperature(lg['val'], y_val)
        for p in parts:
            R[p]['single_det'][m] = softmax_np(lg[p]); R[p]['single_cal'][m] = softmax_np(lg[p] / Ts[m])
            seed = (base + 17 * mi + (0 if p == 'val' else 7919)) if p in ('val', 'test') else EXT_MC_SEED0 + mi
            samples[p].append(head_mc(h, feats[m][p], MC_SAMPLES, seed))
        del h
    for p in parts:
        R[p]['ens_det'] = np.mean([R[p]['single_det'][m] for m in MODELS], 0)
        R[p]['mc'] = mc_stats(np.concatenate(samples[p], 0)); samples[p] = None
    T_det = fit_temperature(np.log(R['val']['ens_det'] + EPS), y_val)
    T = fit_temperature(np.log(R['val']['mc']['probs'] + EPS), y_val)
    lw = LedoitWolf(assume_centered=False).fit(np.concatenate([feats[m]['nvref'] for m in MODELS], 1).astype(np.float64))
    dist = {p: mahalanobis(np.concatenate([feats[m][p] for m in MODELS], 1), lw.location_, lw.precision_) for p in parts}
    dmin, dmax = dist['val'].min(), dist['val'].max()
    for p in parts:
        R[p]['ens_det_cal'] = softmax_np(np.log(R[p]['ens_det'] + EPS) / T_det)
        R[p]['full'] = softmax_np(np.log(R[p]['mc']['probs'] + EPS) / T)            # full framework prediction
        R[p]['hbds'] = np.clip((dist[p] - dmin) / (dmax - dmin + EPS), 0, 1) * 100
    g, rank = fit_gmm(R['val']['hbds'], GMM_N)
    tau_u = float(np.quantile(R['val']['mc']['var'], UNC_QUANTILE))
    for p in parts:
        R[p]['level'] = gmm_levels(g, rank, R[p]['hbds']); R[p]['unc'] = R[p]['mc']['var'] > tau_u
        R[p]['risk'] = 2 * R[p]['level'] + R[p]['unc'].astype(int)
    R['val'].update(y=y_val, ymal=np.isin(y_val, MAL_IDX).astype(int), groups=GROUPS[sp['val']])
    y_test = LABELS[sp['test']]
    R['test'].update(y=y_test, ymal=np.isin(y_test, MAL_IDX).astype(int), groups=GROUPS[sp['test']], rows=sp['test'])
    for ds, d in ext.items():
        R[ds].update(y=_value(d['y'], k), ymal=_value(d['ymal'], k), groups=_value(d['groups'], k))
    R['meta'] = {'T': T, 'T_det': T_det, 'T_single': Ts, 'tau_u': tau_u, 'dmin': dmin, 'dmax': dmax}
    return R

# ---------------------------------------------------------------- metrics
def p_mal(p):
    return p[:, MAL_IDX].sum(1)

def ece(p, y, n_bins=15):
    conf, ok = p.max(1), (p.argmax(1) == y).astype(float); bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return float(e)

def brier(p, y):
    return float(np.mean(np.sum((p - np.eye(p.shape[1])[y]) ** 2, 1)))

def balanced_acc_present(y, p):
    """Balanced accuracy over the classes present (images outside the 7 classes, y = -1, are ignored)."""
    m = y >= 0; yy, pred = y[m], p[m].argmax(1)
    return float(np.mean([np.mean(pred[yy == c] == c) for c in sorted(np.unique(yy))]))

def cls_metrics(y, ymal, p):
    kn = y >= 0; yy, pp = y[kn], p[kn]; present = sorted(np.unique(yy)); pred = pp.argmax(1)
    return {'Accuracy': accuracy_score(yy, pred),
            'Balanced acc.': float(np.mean([np.mean(pred[yy == c] == c) for c in present])),
            'Macro F1': f1_score(yy, pred, labels=present, average='macro', zero_division=0),
            'Macro AUC': float(np.mean([roc_auc_score((yy == c).astype(int), pp[:, c]) for c in present if 0 < (yy == c).sum() < len(yy)])),
            'Malignant AUC': roc_auc_score(ymal, p_mal(p)) if 0 < ymal.sum() < len(ymal) else np.nan,
            'ECE': ece(pp, yy), 'Brier': brier(pp, yy), 'NLL': nll(pp, yy)}

CONFIGS = OrderedDict([(LABEL_OF[m], (lambda R, m=m: R['single_det'][m])) for m in MODELS] +
                      [('Ensemble (soft vote)', lambda R: R['ens_det']),
                       ('Ensemble + MC Dropout', lambda R: R['mc']['probs']),
                       ('Ensemble + MC Dropout + temperature scaling', lambda R: R['full'])])

In [ ]:
N_BOOT = int(os.environ.get('N_BOOT', 2000))

def make_weights(strata, n_boot=N_BOOT, seed=SEED):
    """Stratified cluster bootstrap: (n_boot, n_clusters) resampling counts, stratified by each cluster's majority class."""
    rng = np.random.default_rng(seed); W = np.zeros((n_boot, len(strata)), np.float32)
    for s in np.unique(strata):
        idx = np.where(strata == s)[0]; draws = rng.integers(0, len(idx), size=(n_boot, len(idx)))
        for b in range(n_boot):
            W[b, idx] += np.bincount(draws[b], minlength=len(idx))
    return W

def cluster_setup(groups, y, seed=SEED):
    codes, CL = np.unique(np.asarray(groups).astype(str), return_inverse=True)
    strata = pd.Series(np.asarray(y)).groupby(CL).agg(lambda s: s.value_counts().index[0]).reindex(range(len(codes))).values
    return CL, make_weights(strata, seed=seed)

def ba_draws(y, correct, CL, W, mask=None):
    """Balanced accuracy and its bootstrap draws for one set of predictions (correct = 0/1 per image)."""
    mask = (y >= 0) if mask is None else (mask & (y >= 0))
    point, draws = [], []
    for c in sorted(np.unique(y[mask])):
        m = mask & (y == c)
        num = np.bincount(CL[m], weights=correct[m].astype(float), minlength=W.shape[1])
        den = np.bincount(CL[m], minlength=W.shape[1]).astype(float)
        point.append(num.sum() / den.sum()); bd = W @ den
        draws.append(np.where(bd > 0, (W @ num) / np.maximum(bd, 1e-12), np.nan))
    return float(np.mean(point)), np.nanmean(np.stack(draws), 0)

def paired_ba_change(y, groups, probs_a, probs_b, mask=None):
    """Mean over fold-ensembles of BA(b) - BA(a) with a paired cluster-bootstrap 95% CI (percentage points)."""
    CL, W = cluster_setup(groups, np.where(y >= 0, y, C))
    pa, pb, da, db = [], [], [], []
    for A_, B_ in zip(probs_a, probs_b):
        a, d_a = ba_draws(y, A_.argmax(1) == y, CL, W, mask); b, d_b = ba_draws(y, B_.argmax(1) == y, CL, W, mask)
        pa.append(a); pb.append(b); da.append(d_a); db.append(d_b)
    d = np.mean(db, 0) - np.mean(da, 0)
    return {'before': float(np.mean(pa)), 'after': float(np.mean(pb)), 'change (pp)': 100 * (np.mean(pb) - np.mean(pa)),
            'CI low (pp)': 100 * np.nanpercentile(d, 2.5), 'CI high (pp)': 100 * np.nanpercentile(d, 97.5)}

def weighted_auc_draws(y, s, W_img, chunk=250):
    """AUC under each bootstrap weight vector (rows of W_img), ties counted as one half."""
    order = np.argsort(s, kind='mergesort'); s_, y_ = s[order], np.asarray(y)[order].astype(bool)
    starts = np.r_[0, np.flatnonzero(np.diff(s_)) + 1]; out = []
    for i in range(0, W_img.shape[0], chunk):
        Wc = W_img[i:i + chunk][:, order]
        neg = np.add.reduceat(np.where(~y_, Wc, 0), starts, axis=1); pos = np.add.reduceat(np.where(y_, Wc, 0), starts, axis=1)
        below = np.cumsum(neg, 1) - neg
        out.append((pos * (below + 0.5 * neg)).sum(1) / np.maximum(pos.sum(1) * neg.sum(1), 1e-12))
    return np.concatenate(out)

In [ ]:
GT_COLS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']
GT_MAP = {'MEL': 'mel', 'NV': 'nv', 'BCC': 'bcc', 'AK': 'akiec', 'BKL': 'bkl', 'DF': 'df', 'VASC': 'vasc', 'SCC': 'scc', 'UNK': None}
N_ISIC_SAMPLE = int(os.environ.get('N_ISIC_SAMPLE', 5000))   # size of the stratified ISIC 2019 sample used in the study
IND_DIR = PROJECT_OUT / 'isic2019_audit' / 'independent_crop_validation'

def load_isic2019_tables():
    gt = pd.read_csv(find_file(ISIC2019_DIR, 'ISIC_2019_Training_GroundTruth.csv'))
    gt['orig'] = gt[[c for c in GT_COLS if c in gt]].idxmax(1)
    gt = gt.drop_duplicates('image').set_index('image')
    meta = pd.read_csv(find_file(ISIC2019_DIR, 'ISIC_2019_Training_Metadata.csv')).drop_duplicates('image').set_index('image')
    return gt, meta

def prepare_isic2019_sample(ham_ids):
    """Stratified random sample of 5,000 ISIC 2019 images not in HAM10000, cached at 224 x 224 (reused if present)."""
    gt, meta = load_isic2019_tables()
    npy, js = DIRS['cache'] / f'images_ext_ISIC2019_{IMG_SIZE}.npy', DIRS['cache'] / f'images_ext_ISIC2019_{IMG_SIZE}.json'
    pool = gt[~gt.index.isin(set(ham_ids)) & (gt['orig'] != 'UNK')]
    if js.exists():
        ids = [Path(p).stem for p in json.load(open(js))]
        paths = json.load(open(js))
    else:
        from sklearn.model_selection import train_test_split
        n = min(N_ISIC_SAMPLE, len(pool))
        ids = list(pool.index) if n == len(pool) else \
            list(train_test_split(pool.index.values, train_size=n, stratify=pool['orig'].values, random_state=SEED)[0])
        index = image_index(ISIC2019_DIR)
        missing = [i for i in ids if i not in index]
        if missing:
            raise FileNotFoundError(f'{len(missing)} ISIC 2019 images not found under {ISIC2019_DIR}, e.g. {missing[:3]}')
        paths = [index[i] for i in ids]
    imgs = build_image_cache(paths, npy, js, desc='caching ISIC 2019 sample')
    df = pd.DataFrame({'image_id': ids})
    df['dx'] = gt.reindex(ids)['orig'].map(GT_MAP).values
    df['label'] = df['dx'].map(CLASS2IDX).fillna(-1).astype(int)                 # scc = -1 (no HAM10000 class)
    df['lesion_id'] = meta.reindex(ids)['lesion_id'].values
    df['group'] = np.where(df['lesion_id'].notna(), df['lesion_id'].astype(str), 'IMG_' + df['image_id'])
    df['path'] = [str(p) for p in paths]
    return df, imgs, len(pool)

def dhash64(img, size=8):
    g = cv2.resize(cv2.cvtColor(np.ascontiguousarray(img), cv2.COLOR_RGB2GRAY), (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = (g[:, 1:] > g[:, :-1]).ravel()
    return np.uint64(int(''.join('1' if b else '0' for b in bits), 2))

def orientations(img):
    for r in range(4):
        rot = np.rot90(img, r)
        yield rot
        yield rot[:, ::-1]

def _popcount(x):
    if hasattr(np, 'bitwise_count'):
        return np.bitwise_count(x)
    lut = np.array([bin(i).count('1') for i in range(256)], np.uint8)
    return lut[x.view(np.uint8).reshape(*x.shape, 8)].sum(-1)

def near_duplicates(cand_hashes, ref_hashes, max_dist=4, chunk=512):
    """cand_hashes (n, 8 orientations); returns a boolean mask of candidates within max_dist of any reference."""
    hit = np.zeros(len(cand_hashes), bool); ref = np.asarray(ref_hashes, np.uint64)
    for i in range(0, len(cand_hashes), chunk):
        c = np.asarray(cand_hashes[i:i + chunk], np.uint64)
        d = _popcount(c[:, :, None] ^ ref[None, None, :])
        hit[i:i + chunk] = (d <= max_dist).any((1, 2))
    return hit

def prepare_independent_set(isic_df, ham_ids, ham_imgs, isic_imgs):
    """Second external set: remaining ISIC 2019 images in the 7 classes, lesion-disjoint and not near-duplicates.

    Reuses the study's manifest if present; otherwise rebuilds it with the documented label-free rules
    (a rebuilt set can differ slightly in size from the 3,134 images used in the study).
    """
    man_path = IND_DIR / 'independent_image_manifest.csv'
    gt, meta = load_isic2019_tables()
    if man_path.exists():
        man = pd.read_csv(man_path)
        man = man[man['included'].astype(str).str.lower() == 'true'].reset_index(drop=True)
    else:
        print('Building the independent set (label-free near-duplicate screening) ...')
        used = set(isic_df['image_id']) | set(ham_ids)
        pool = gt[~gt.index.isin(used) & gt['orig'].isin(['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC'])].copy()
        pool['lesion_id'] = meta.reindex(pool.index)['lesion_id'].values
        taken = set(isic_df['lesion_id'].dropna().astype(str)) | set(meta.reindex(list(ham_ids))['lesion_id'].dropna().astype(str))
        pool = pool[~pool['lesion_id'].astype(str).isin(taken) | pool['lesion_id'].isna()]
        index = image_index(ISIC2019_DIR)
        pool = pool[pool.index.isin(index)]
        ref = [dhash64(im) for im in tqdm(ham_imgs, desc='hashing HAM10000')] + [dhash64(im) for im in tqdm(isic_imgs, desc='hashing ISIC sample')]
        cand_imgs = [read_rgb_224(index[i]) for i in tqdm(pool.index, desc='loading candidates')]
        cand = np.array([[dhash64(o) for o in orientations(im)] for im in cand_imgs], np.uint64)
        dup = near_duplicates(cand, ref)
        man = pd.DataFrame({'image_id': pool.index, 'dx': pool['orig'].map(GT_MAP).values, 'lesion_id': pool['lesion_id'].values,
                            'path': [str(index[i]) for i in pool.index], 'included': ~dup})
        IND_DIR.mkdir(parents=True, exist_ok=True); man.to_csv(man_path, index=False)
        man = man[man['included']].reset_index(drop=True)
    cache_npy, cache_js = IND_DIR / 'derived' / f'independent_images_{IMG_SIZE}.npy', IND_DIR / 'derived' / f'independent_images_{IMG_SIZE}.json'
    if cache_js.exists():
        listed = json.load(open(cache_js)); row_of = {Path(p).stem: i for i, p in enumerate(listed)}
        all_imgs = np.load(cache_npy, mmap_mode='r'); rows = np.array([row_of[i] for i in man['image_id'].astype(str)])
        imgs = all_imgs if (len(rows) == len(all_imgs) and (rows == np.arange(len(rows))).all()) else np.asarray(all_imgs[rows])
        if 'path' not in man:
            man['path'] = [listed[r] for r in rows]
    else:
        if 'path' not in man:
            index = image_index(ISIC2019_DIR); man['path'] = [str(index[i]) for i in man['image_id'].astype(str)]
        imgs = build_image_cache(man['path'].tolist(), cache_npy, cache_js, desc='caching independent set')
    man['label'] = man['dx'].map(CLASS2IDX).fillna(-1).astype(int)
    man['group'] = np.where(man['lesion_id'].notna(), man['lesion_id'].astype(str), 'IMG_' + man['image_id'].astype(str))
    return man, imgs

def ext_feat_path(name, k, m, n):
    """Feature file of an external set for fold k / model m (n = number of images, used to match cached files)."""
    if name == 'ISIC 2019':
        return fold_dir(k) / f'external_ISIC2019_{m}.npy'
    if name == 'Independent ISIC set':
        for d in sorted((IND_DIR / 'features').glob('*')):
            f = d / f'baseline_fold{k}_{m}.npy'
            if f.exists() and np.load(f, mmap_mode='r').shape[0] == n:
                return f
        return IND_DIR / 'features' / 'rebuilt' / f'baseline_fold{k}_{m}.npy'
    raise KeyError(name)

def ensure_external_features(name, images, folds):
    """Extract backbone features of an external set with every fold model (cached files are reused, never overwritten)."""
    for k in folds:
        for m in MODELS:
            p = ext_feat_path(name, k, m, len(images))
            if p.exists():
                if np.load(p, mmap_mode='r').shape[0] != len(images):
                    raise RuntimeError(f'{p} has a different number of rows than the image set - check the caches.')
                continue
            p.parent.mkdir(parents=True, exist_ok=True)
            model = load_model(k, m)
            np.save(p, extract_features(model, images))
            del model
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()

def ext_entry(name, df):
    return dict(feats=lambda k, m, _n=name, _len=len(df): np.load(ext_feat_path(_n, k, m, _len)), y=df['label'].values,
                ymal=df['dx'].isin(MALIGNANT + ['scc']).astype(int).values, groups=df['group'].values)

In [ ]:
# Border detector and crop policy (rule fixed before the confirmatory experiment)
DARK_LEVEL, BORDER_MIN_FRAC, CORNER_BOX, CORNER_DARK_MIN, MIN_DARK_CORNERS = 40, 0.10, 16, 0.60, 3
CROP_PURITY, CROP_MIN_SIDE = 0.995, 0.40

def border_measure(img224):
    """Share of the image covered by dark regions touching the edge, number of dark corners, edge mask."""
    g = cv2.GaussianBlur(cv2.cvtColor(np.ascontiguousarray(img224), cv2.COLOR_RGB2GRAY), (3, 3), 0)
    dark = (g < DARK_LEVEL).astype(np.uint8)
    _, lab = cv2.connectedComponents(dark, connectivity=8)
    el = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])); el = el[el != 0]
    edge = np.isin(lab, el); b = CORNER_BOX
    corners = [dark[:b, :b], dark[:b, -b:], dark[-b:, :b], dark[-b:, -b:]]
    return float(edge.mean()), int(sum(c.mean() >= CORNER_DARK_MIN for c in corners)), edge

def is_bordered(img224):
    cov, corners, _ = border_measure(img224)
    return cov >= BORDER_MIN_FRAC and corners >= MIN_DARK_CORNERS

def field_crop_box(img224):
    """Largest centred square inside the field of view (224-scale coordinates)."""
    _, _, edge = border_measure(img224); field = ~edge; H, W_ = field.shape
    if field.mean() < 0.2:
        return 0, 0, W_, H, 'fallback: field of view too small'
    ys, xs = np.nonzero(field); cy, cx = int(round(ys.mean())), int(round(xs.mean()))
    for h in range(min(H, W_) // 2, 0, -1):
        y0, y1, x0, x1 = max(cy - h, 0), min(cy + h, H), max(cx - h, 0), min(cx + h, W_)
        if field[y0:y1, x0:x1].mean() >= CROP_PURITY:
            if min(y1 - y0, x1 - x0) < CROP_MIN_SIDE * min(H, W_):
                return 0, 0, W_, H, 'fallback: crop would be too small'
            return x0, y0, x1, y1, 'cropped'
    return 0, 0, W_, H, 'fallback: no clean square found'

def crop_original(path, box):
    """Apply a 224-scale crop box to the full-resolution image and resize the crop to 224 x 224."""
    x0, y0, x1, y1 = box[:4]; raw = read_rgb(path); H, W_ = raw.shape[:2]; sx, sy = W_ / IMG_SIZE, H / IMG_SIZE
    c = raw[int(round(y0 * sy)):int(round(y1 * sy)), int(round(x0 * sx)):int(round(x1 * sx))]
    return cv2.resize(c, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

In [ ]:
ham, LABELS, GROUPS = load_ham()
SPLITS = load_splits(); FOLDS = sorted(SPLITS)
HAM_IMGS = np.load(DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.npy', mmap_mode='r')
print(f'{len(ham)} images, {len(set(GROUPS))} lesions, folds {FOLDS}')

In [ ]:
FA_DIR = PROJECT_OUT / 'failure_analysis'; (FA_DIR / 'features').mkdir(parents=True, exist_ok=True)
RUN_PROBES = os.environ.get('RUN_PROBES', '1') == '1'
RECOVERED_CSV = os.environ.get('RECOVERED_BORDERED_CSV')        # optional: image_id list of the recovered bordered images

for k in FOLDS:
    ensure_fold_cache(k, HAM_IMGS, LABELS, SPLITS)
isic, ISIC_IMGS, _ = prepare_isic2019_sample(ham['image_id'])
ensure_external_features('ISIC 2019', ISIC_IMGS, FOLDS)
ind, IND_IMGS = prepare_independent_set(isic, ham['image_id'], HAM_IMGS, ISIC_IMGS)
ensure_external_features('Independent ISIC set', IND_IMGS, FOLDS)

def cached_features(tag, images, k, m):
    """Features of modified images for fold k / model m, cached in outputs/failure_analysis/features."""
    p = FA_DIR / 'features' / f'{tag}_fold{k}_{m}.npy'
    if p.exists() and np.load(p, mmap_mode='r').shape[0] == len(images):
        return np.load(p)
    model = load_model(k, m); f = extract_features(model, images); np.save(p, f); del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return f

## 1. Verification: HAM10000 duplicates in the external sample

In [ ]:
ham_hash = np.array([dhash64(im) for im in tqdm(HAM_IMGS, desc='hashing HAM10000')], np.uint64)
isic_hash8 = np.array([[dhash64(o) for o in orientations(np.asarray(im))] for im in tqdm(ISIC_IMGS, desc='hashing ISIC sample')], np.uint64)
dup = near_duplicates(isic_hash8, ham_hash)
print(f'ISIC 2019 sample images that are near-duplicates of a HAM10000 image (dHash distance <= 4, 8 orientations): {int(dup.sum())}')
print(f'ISIC identifiers also present in HAM10000: {len(set(isic.image_id) & set(ham.image_id))}')

## 2. Dermoscope borders: prevalence and effect

In [ ]:
B_HAM = np.array([is_bordered(np.asarray(im)) for im in tqdm(HAM_IMGS, desc='borders HAM10000')])
B_ISIC = np.array([is_bordered(np.asarray(im)) for im in tqdm(ISIC_IMGS, desc='borders ISIC 2019')])
B_IND = np.array([is_bordered(np.asarray(im)) for im in tqdm(IND_IMGS, desc='borders independent')])
prev = pd.DataFrame({'images': [len(B_HAM), len(B_ISIC), len(B_IND)], 'bordered': [B_HAM.sum(), B_ISIC.sum(), B_IND.sum()]},
                    index=['HAM10000', 'ISIC 2019 sample', 'Independent ISIC set'])
prev['bordered (%)'] = (100 * prev['bordered'] / prev['images']).round(1); display(prev)
prev.to_csv(FA_DIR / 'border_prevalence.csv')

## 3. Border-crop policy

Only images flagged by the label-free detector are cropped (from the full-resolution original); all other images are unchanged.
Unchanged images keep exactly the same probabilities, so the comparison is paired.

In [ ]:
def crop_set(images, paths, flags, tag):
    idx = np.where(flags)[0]; boxes = {int(i): field_crop_box(np.asarray(images[i])) for i in idx}
    done = np.array([i for i in idx if boxes[int(i)][4] == 'cropped'], int)
    imgs = np.stack([crop_original(paths[i], boxes[int(i)]) for i in done]) if len(done) else np.zeros((0, IMG_SIZE, IMG_SIZE, 3), np.uint8)
    print(f'{tag}: {len(idx)} bordered images, {len(done)} cropped')
    return done, imgs

CROP_ISIC_IDX, CROP_ISIC_IMGS = crop_set(ISIC_IMGS, isic['path'].tolist(), B_ISIC, 'ISIC 2019')
CROP_IND_IDX, CROP_IND_IMGS = crop_set(IND_IMGS, ind['path'].tolist(), B_IND, 'Independent set')

def replaced(name, n, rows, tag, crop_imgs):
    def f(k, m):
        base = np.load(ext_feat_path(name, k, m, n)).copy()
        if len(rows):
            base[rows] = cached_features(tag, crop_imgs, k, m)
        return base
    return f

EXT = OrderedDict()
EXT['ISIC 2019'] = ext_entry('ISIC 2019', isic)
EXT['ISIC 2019 + crop'] = {**ext_entry('ISIC 2019', isic), 'feats': replaced('ISIC 2019', len(isic), CROP_ISIC_IDX, 'isic_crop', CROP_ISIC_IMGS)}
EXT['Independent ISIC set'] = ext_entry('Independent ISIC set', ind)
EXT['Independent ISIC set + crop'] = {**ext_entry('Independent ISIC set', ind),
                                      'feats': replaced('Independent ISIC set', len(ind), CROP_IND_IDX, 'ind_crop', CROP_IND_IMGS)}

### Optional confirmatory set

In the study, the crop policy was confirmed on 554 bordered images that an over-strict near-duplicate filter had excluded from both
external sets. To repeat that test, set `RECOVERED_BORDERED_CSV` to a CSV with an `image_id` column listing those images.

In [ ]:
REC = None
if RECOVERED_CSV and Path(RECOVERED_CSV).exists():
    gt, meta_isic = load_isic2019_tables(); index = image_index(ISIC2019_DIR)
    ids = pd.read_csv(RECOVERED_CSV)['image_id'].astype(str).tolist()
    REC = pd.DataFrame({'image_id': ids, 'dx': gt.reindex(ids)['orig'].map(GT_MAP).values, 'lesion_id': meta_isic.reindex(ids)['lesion_id'].values,
                        'path': [str(index[i]) for i in ids]})
    REC['label'] = REC['dx'].map(CLASS2IDX).fillna(-1).astype(int)
    REC['group'] = np.where(REC['lesion_id'].notna(), REC['lesion_id'].astype(str), 'IMG_' + REC['image_id'])
    REC_IMGS = build_image_cache(REC['path'].tolist(), FA_DIR / 'recovered_224.npy', FA_DIR / 'recovered_224.json', 'caching recovered set')
    B_REC = np.array([is_bordered(np.asarray(im)) for im in REC_IMGS])
    CROP_REC_IDX, CROP_REC_IMGS = crop_set(REC_IMGS, REC['path'].tolist(), B_REC, 'Recovered bordered set')
    base_rec = lambda k, m: cached_features('recovered', REC_IMGS, k, m)
    def rec_crop(k, m):
        f = base_rec(k, m).copy()
        if len(CROP_REC_IDX):
            f[CROP_REC_IDX] = cached_features('recovered_crop', CROP_REC_IMGS, k, m)
        return f
    entry = dict(y=REC['label'].values, ymal=REC['dx'].isin(MALIGNANT + ['scc']).astype(int).values, groups=REC['group'].values)
    EXT['Recovered bordered set'] = {**entry, 'feats': base_rec}
    EXT['Recovered bordered set + crop'] = {**entry, 'feats': rec_crop}
else:
    print('Recovered bordered set not provided - confirmatory test skipped.')

## 4. Shades-of-Gray colour constancy

In [ ]:
def shades_of_gray(img, p=6):
    x = np.asarray(img, np.float64)
    e = np.power(np.mean(np.power(x, p), axis=(0, 1)), 1 / p); e = e / (np.linalg.norm(e) + EPS)
    return np.clip(x / (e * np.sqrt(3) + EPS), 0, 255).astype(np.uint8)

SOG_ISIC = np.stack([shades_of_gray(im) for im in tqdm(ISIC_IMGS, desc='Shades of Gray (ISIC)')])
EXT['ISIC 2019 + Shades of Gray'] = {**ext_entry('ISIC 2019', isic), 'feats': lambda k, m: cached_features('isic_sog', SOG_ISIC, k, m)}
SOG_HAM = {}
def ham_sog_feats(k, m):
    if k not in SOG_HAM:
        SOG_HAM[k] = np.stack([shades_of_gray(HAM_IMGS[r]) for r in SPLITS[k]['test']])
    return cached_features(f'ham_test_sog_k{k}', SOG_HAM[k], k, m)
EXT['HAM10000 test + Shades of Gray'] = dict(feats=ham_sog_feats, y=lambda k: LABELS[SPLITS[k]['test']],
                                             ymal=lambda k: np.isin(LABELS[SPLITS[k]['test']], MAL_IDX).astype(int),
                                             groups=lambda k: GROUPS[SPLITS[k]['test']])

## 5. Rebuild the framework with all variants

In [ ]:
FOLD = {k: build_fold(k, EXT) for k in tqdm(FOLDS, desc='rebuilding the framework')}
for base, var in [('ISIC 2019', 'ISIC 2019 + crop'), ('Independent ISIC set', 'Independent ISIC set + crop')]:
    unchanged = np.ones(len(FOLD[FOLDS[0]][base]['y']), bool)
    unchanged[CROP_ISIC_IDX if base == 'ISIC 2019' else CROP_IND_IDX] = False
    diff = max(np.abs(FOLD[k][base]['full'][unchanged] - FOLD[k][var]['full'][unchanged]).max() if unchanged.any() else 0 for k in FOLDS)
    print(f'{base}: max |probability change| on unchanged images = {diff:.1e}')

## 6. Border-crop results

In [ ]:
def paired(base, var, mask=None):
    y, g = FOLD[FOLDS[0]][base]['y'], FOLD[FOLDS[0]][base]['groups']
    return paired_ba_change(y, g, [FOLD[k][base]['full'] for k in FOLDS], [FOLD[k][var]['full'] for k in FOLDS], mask)

crop_rows = []
if REC is not None:
    r = paired('Recovered bordered set', 'Recovered bordered set + crop', B_REC)
    crop_rows.append(['Crop only when a border is detected (rule fixed in advance)', 'Recovered bordered images', int(B_REC.sum()), r, 'confirmatory'])
r = paired('ISIC 2019', 'ISIC 2019 + crop')
crop_rows.append(['Crop only when a border is detected', 'Full ISIC 2019 sample', int((isic.label >= 0).sum()), r, 'exploratory'])
r = paired('ISIC 2019', 'ISIC 2019 + crop', B_ISIC)
crop_rows.append(['Same policy, bordered images only', 'ISIC 2019 bordered images', int((B_ISIC & (isic.label.values >= 0)).sum()), r, 'exploratory'])
r = paired('Independent ISIC set', 'Independent ISIC set + crop')
crop_rows.append(['Same policy', f'Independent ISIC set ({int(B_IND.sum())} bordered)', int((ind.label >= 0).sum()), r, 'independent set'])
CROP = pd.DataFrame([{'experiment': e, 'test_set': s, 'images': n, 'balanced_acc_before_pct': round(100 * r['before'], 1),
                      'balanced_acc_after_pct': round(100 * r['after'], 1), 'gain_pp': round(r['change (pp)'], 2),
                      'gain_ci95_low_pp': round(r['CI low (pp)'], 2), 'gain_ci95_high_pp': round(r['CI high (pp)'], 2), 'note': note}
                     for e, s, n, r, note in crop_rows])
CROP.to_csv(RESULTS_DIR / 'border_crop_results.csv', index=False); display(CROP)

## 7. Colour constancy and prior adjustment

In [ ]:
def em_prior_adjust(p, prior_src, n_iter=100, tol=1e-6):
    """Saerens et al. (2002): re-estimate the class priors of a new data set without labels."""
    prior = prior_src.copy()
    for _ in range(n_iter):
        q = p * (prior / prior_src); q /= q.sum(1, keepdims=True)
        new = q.mean(0)
        if np.abs(new - prior).max() < tol:
            break
        prior = new
    return q

other = []
r = paired('ISIC 2019', 'ISIC 2019 + Shades of Gray'); other.append({'experiment': 'Shades-of-Gray colour constancy', 'test set': 'ISIC 2019', **r})
ham_change = [100 * (balanced_acc_present(FOLD[k]['HAM10000 test + Shades of Gray']['y'], FOLD[k]['HAM10000 test + Shades of Gray']['full'])
                     - balanced_acc_present(FOLD[k]['test']['y'], FOLD[k]['test']['full'])) for k in FOLDS]
other.append({'experiment': 'Shades-of-Gray colour constancy', 'test set': 'HAM10000 test folds', 'change (pp)': float(np.mean(ham_change))})
EM = {k: em_prior_adjust(FOLD[k]['ISIC 2019']['full'], FOLD[k]['val']['full'].mean(0)) for k in FOLDS}
y0, g0 = FOLD[FOLDS[0]]['ISIC 2019']['y'], FOLD[FOLDS[0]]['ISIC 2019']['groups']
r = paired_ba_change(y0, g0, [FOLD[k]['ISIC 2019']['full'] for k in FOLDS], [EM[k] for k in FOLDS])
other.append({'experiment': 'EM prior adjustment', 'test set': 'ISIC 2019', **r})
OTHER = pd.DataFrame(other).round(3); display(OTHER); OTHER.to_csv(FA_DIR / 'colour_and_prior_interventions.csv', index=False)

## 8. Features versus decision layer: linear probes

* **HAM10000 probe:** an L2-regularised logistic regression (balanced class weights) on the concatenated frozen features of each
  fold's training images, with the regularisation strength chosen on HAM10000 validation data.
* **ISIC-trained probe (diagnostic only):** the same model trained on ISIC 2019 labels with nested, lesion-grouped cross-validation.
  It shows how much external accuracy the frozen features could support if only the decision layer were adapted; it is not a result
  of the framework.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
C_GRID = [0.01, 0.1, 1.0]

def probe(Cv):
    return make_pipeline(StandardScaler(), LogisticRegression(C=Cv, class_weight='balanced', max_iter=2000))

PROBE = []
if RUN_PROBES:
    kn = isic.label.values >= 0; y_isic = isic.label.values; g_isic = isic.group.values
    for k in tqdm(FOLDS, desc='linear probes'):
        sp = SPLITS[k]
        Xtr = np.concatenate([cached_features(f'ham_train_k{k}', np.asarray(HAM_IMGS[sp['train']]), k, m)
                              for m in MODELS], 1).astype(np.float32)
        Xva = np.concatenate([np.load(feat_path(k, m))['val'] for m in MODELS], 1).astype(np.float32)
        Xis = np.concatenate([np.load(ext_feat_path('ISIC 2019', k, m, len(isic))) for m in MODELS], 1).astype(np.float32)
        scores = {Cv: balanced_accuracy_score(LABELS[sp['val']], probe(Cv).fit(Xtr, LABELS[sp['train']]).predict(Xva)) for Cv in C_GRID}
        Cbest = max(scores, key=scores.get)
        p_ham = probe(Cbest).fit(Xtr, LABELS[sp['train']]).predict(Xis[kn])
        oof = np.full(kn.sum(), -1)
        outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        Xk, yk, gk = Xis[kn], y_isic[kn], g_isic[kn]
        for tr, te in outer.split(Xk, yk, gk):
            inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
            itr, iva = next(inner.split(Xk[tr], yk[tr], gk[tr]))
            ci = max(C_GRID, key=lambda Cv: balanced_accuracy_score(yk[tr][iva], probe(Cv).fit(Xk[tr][itr], yk[tr][itr]).predict(Xk[tr][iva])))
            oof[te] = probe(ci).fit(Xk[tr], yk[tr]).predict(Xk[te])
        PROBE.append({'fold': k, 'framework BA': balanced_acc_present(y_isic, FOLD[k]['ISIC 2019']['full']),
                      'HAM10000 probe BA': balanced_accuracy_score(yk, p_ham), 'ISIC-trained probe BA (diagnostic)': balanced_accuracy_score(yk, oof)})
    PROBE = pd.DataFrame(PROBE); display(PROBE.round(4)); display(PROBE.drop(columns='fold').mean().round(4).to_frame('mean'))
    PROBE.to_csv(FA_DIR / 'linear_probes.csv', index=False)
else:
    print('Linear probes skipped (RUN_PROBES=0).')

## 9. Retraining experiments (not repeated here)

The study also retrained ConvNeXt-Base on fold 0 three times with the original recipe and three times with stronger augmentation
(synthetic dermoscope borders, random resized crops, white-balance shifts, JPEG compression). These runs showed large run-to-run
variation in external accuracy (0.481–0.502 for identical settings) and no benefit from the stronger augmentation. They can be
repeated with notebook 02 by changing the seed and the augmentation pipeline; they are not re-run here because each run takes hours.

## Summary

In [ ]:
summary = [('Duplicates of HAM10000 images in the external sample', f'{int(dup.sum())} near-duplicates found')]
for _, r in CROP.iterrows():
    summary.append((f"Border crop - {r['test_set']}", f"{r['balanced_acc_before_pct']}% -> {r['balanced_acc_after_pct']}% "
                    f"({r['gain_pp']:+.2f} pp, CI {r['gain_ci95_low_pp']:+.2f} to {r['gain_ci95_high_pp']:+.2f})"))
for _, r in OTHER.iterrows():
    summary.append((f"{r['experiment']} - {r['test set']}", f"{r['change (pp)']:+.2f} pp"))
if len(PROBE):
    summary.append(('Linear probe trained on HAM10000', f"{PROBE['HAM10000 probe BA'].mean():.3f} (framework {PROBE['framework BA'].mean():.3f})"))
    summary.append(('Linear probe trained on ISIC 2019 labels (diagnostic)', f"{PROBE['ISIC-trained probe BA (diagnostic)'].mean():.3f}"))
SUMMARY = pd.DataFrame(summary, columns=['experiment', 'result']); display(SUMMARY)
SUMMARY.to_csv(FA_DIR / 'failure_analysis_summary.csv', index=False)